# Parallel Consistency Diagnostic

This notebook tests whether the same ML predictor parameter case produces the same result when run:

1. in a plain Python `for` loop,
2. through `joblib.Parallel(..., n_jobs=1, backend="loky")`, and
3. optionally through `joblib.Parallel(..., n_jobs>1, backend="loky")`.

The point is to isolate the parallelization question from the research notebook state. Run this notebook after restarting the kernel.

## 0. Clean Imports

This cell shuts down reusable `loky` workers and reloads the local modules from disk. This matters because notebook kernels can keep old function objects in memory, while `loky` workers import code separately.

In [5]:
from pathlib import Path
import os
import sys
import time
import importlib
import itertools

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor

# Make sure this notebook imports from its own folder.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "ML_predictor":
    PROJECT_DIR = PROJECT_DIR / "research" / "Mateo" / "ML_predictor"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Kill any reusable loky workers that may have imported older code.
get_reusable_executor().shutdown(wait=True)

import ML_predictor
import run_grid_case
import run_single_case

importlib.reload(ML_predictor)
importlib.reload(run_grid_case)
importlib.reload(run_single_case)

from ML_predictor import MLPredictorStrategy
from run_grid_case import run_grid_case
from src.backtest.vector_backtester__Loss_rate_and_bm_rates import PortfolioVectorEngine

print("Imported from:")
print("ML_predictor:", Path(ML_predictor.__file__).resolve())
print("run_grid_case:", Path(run_grid_case.__code__.co_filename).resolve())
print("cwd:", Path.cwd())

Imported from:
ML_predictor: /Users/mateo/Bluegrey/research/Mateo/ML_predictor/ML_predictor.py
run_grid_case: /Users/mateo/Bluegrey/research/Mateo/ML_predictor/run_grid_case.py
cwd: /Users/mateo/Bluegrey/research/Mateo/ML_predictor


## 1. Define The Cases

These are the same-parameter combinations that overlapped in your notebook. They are intentionally small enough to inspect, but still cover the cases where you saw disagreement.

In [6]:
FIXED = dict(
    symbol="ECH",
    asset_class="STK",
    start_date="2025-01-01",
    end_date="2025-07-01",
    signal_threshold=0.002,
    position_size=1.0,
    initial_capital=1_000_000,
    max_iter=5000,
    execution_delay=1,
    verbose=False,
    show_progress=False,
)

# Set of cases from the inconsistent notebook comparisons.
CASES = []

# Matches the individual loop: n_features varies, training_years fixed at 10.
for nf in [6, 8, 10, 12, 14, 16]:
    CASES.append(dict(n_features=nf, training_years=10))

# Matches the two n_jobs=1 grids: n_features fixed at 12, training_years varies.
for ty in [4, 6, 8, 10, 12, 14]:
    case = dict(n_features=12, training_years=ty)
    if case not in CASES:
        CASES.append(case)

cases_df = pd.DataFrame(CASES)
cases_df

,n_features,training_years
0,6,10
1,8,10
2,10,10
3,12,10
4,14,10
5,16,10
6,12,4
7,12,6
8,12,8
9,12,12


## 2. Comparison Helpers

`run_grid_case` already returns numeric statistics without plotting or creating a tearsheet, so it is the cleanest existing function for this diagnostic. We call the same function in every mode.

In [7]:
RESULT_COLUMNS = [
    "case_id",
    "symbol",
    "start_date",
    "end_date",
    "signal_threshold",
    "n_features",
    "training_years",
    "position_size",
    "max_iter",
    "execution_delay",
    "net_return_pct",
    "gross_return_pct",
    "final_net_equity",
    "final_gross_equity",
]

KEY_COLUMNS = [
    "symbol",
    "start_date",
    "end_date",
    "signal_threshold",
    "n_features",
    "training_years",
    "position_size",
    "max_iter",
    "execution_delay",
]

VALUE_COLUMNS = [
    "net_return_pct",
    "gross_return_pct",
    "final_net_equity",
    "final_gross_equity",
]


def evaluate_case(case_id, case, fixed=FIXED):
    params = {**fixed, **case}
    result = run_grid_case(**params)
    if result is None:
        raise RuntimeError(f"No weights produced for case_id={case_id}, params={params}")
    # run_grid_case currently does not include execution_delay in its returned params.
    result["execution_delay"] = params["execution_delay"]
    result["case_id"] = case_id
    return result


def normalize_results(records):
    df = pd.DataFrame(records)
    missing = [c for c in RESULT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected result columns: {missing}")
    df = df[RESULT_COLUMNS].sort_values("case_id").reset_index(drop=True)
    for col in VALUE_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="raise")
    return df


def run_serial(cases=CASES):
    records = []
    start = time.time()
    for case_id, case in enumerate(cases):
        print(f"serial case {case_id}: {case}")
        records.append(evaluate_case(case_id, case))
    print(f"serial elapsed: {(time.time() - start) / 60:.1f} min")
    return normalize_results(records)


def run_loky(cases=CASES, n_jobs=1):
    get_reusable_executor().shutdown(wait=True)
    start = time.time()
    records = Parallel(n_jobs=n_jobs, backend="loky", verbose=50)(
        delayed(evaluate_case)(case_id, case)
        for case_id, case in enumerate(cases)
    )
    print(f"loky n_jobs={n_jobs} elapsed: {(time.time() - start) / 60:.1f} min")
    return normalize_results(records)


def compare_results(left, right, left_name="left", right_name="right", atol=1e-6, rtol=1e-6):
    left = left.sort_values("case_id").reset_index(drop=True)
    right = right.sort_values("case_id").reset_index(drop=True)

    try:
        assert_frame_equal(
            left[KEY_COLUMNS + VALUE_COLUMNS],
            right[KEY_COLUMNS + VALUE_COLUMNS],
            check_dtype=False,
            atol=atol,
            rtol=rtol,
        )
        print(f"PASS: {left_name} and {right_name} match within atol={atol}, rtol={rtol}.")
        return pd.DataFrame()
    except AssertionError as exc:
        print(f"FAIL: {left_name} and {right_name} differ.")
        print(exc)

    merged = left.merge(
        right,
        on=["case_id"] + KEY_COLUMNS,
        suffixes=(f"_{left_name}", f"_{right_name}"),
        how="outer",
        indicator=True,
    )

    for col in VALUE_COLUMNS:
        merged[f"{col}_diff"] = merged[f"{col}_{right_name}"] - merged[f"{col}_{left_name}"]

    diff_cols = ["case_id"] + KEY_COLUMNS + [
        item
        for col in VALUE_COLUMNS
        for item in (f"{col}_{left_name}", f"{col}_{right_name}", f"{col}_diff")
    ] + ["_merge"]

    diffs = merged[diff_cols]
    display(diffs)
    return diffs

## 3. Test A: Serial Twice

Before testing parallelism, test whether the same serial code is deterministic when called twice. If this fails, the problem is not `Parallel`; the backtest itself is varying between calls.

In [8]:
serial_1 = run_serial(CASES)
serial_1

serial case 0: {'n_features': 6, 'training_years': 10}
serial case 1: {'n_features': 8, 'training_years': 10}
serial case 2: {'n_features': 10, 'training_years': 10}
serial case 3: {'n_features': 12, 'training_years': 10}
serial case 4: {'n_features': 14, 'training_years': 10}
serial case 5: {'n_features': 16, 'training_years': 10}
serial case 6: {'n_features': 12, 'training_years': 4}
serial case 7: {'n_features': 12, 'training_years': 6}
serial case 8: {'n_features': 12, 'training_years': 8}
serial case 9: {'n_features': 12, 'training_years': 12}
serial case 10: {'n_features': 12, 'training_years': 14}
serial elapsed: 29.2 min


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469640,11.623609,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265471,7.624570,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,15.132485,17.690278,1.151325e+06,1.176903e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,16.317112,19.003864,1.163171e+06,1.190039e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-14.717779,-11.835155,8.528222e+05,8.816485e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-2.459850,0.296069,9.754015e+05,1.002961e+06
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-4.505327,-3.368760,9.549467e+05,9.663124e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-10.174324,-9.274388,8.982568e+05,9.072561e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-4.578179,-2.361891,9.542182e+05,9.763811e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,4.091104,6.599758,1.040911e+06,1.065998e+06


In [9]:
serial_2 = run_serial(CASES)
serial_2

serial case 0: {'n_features': 6, 'training_years': 10}
serial case 1: {'n_features': 8, 'training_years': 10}
serial case 2: {'n_features': 10, 'training_years': 10}
serial case 3: {'n_features': 12, 'training_years': 10}
serial case 4: {'n_features': 14, 'training_years': 10}
serial case 5: {'n_features': 16, 'training_years': 10}
serial case 6: {'n_features': 12, 'training_years': 4}
serial case 7: {'n_features': 12, 'training_years': 6}
serial case 8: {'n_features': 12, 'training_years': 8}
serial case 9: {'n_features': 12, 'training_years': 12}
serial case 10: {'n_features': 12, 'training_years': 14}
serial elapsed: 28.1 min


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469640,11.623609,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265471,7.624570,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,15.132485,17.690278,1.151325e+06,1.176903e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,16.317112,19.003864,1.163171e+06,1.190039e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-14.717779,-11.835155,8.528222e+05,8.816485e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-2.459850,0.296069,9.754015e+05,1.002961e+06
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-4.505327,-3.368760,9.549467e+05,9.663124e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-10.174324,-9.274388,8.982568e+05,9.072561e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-4.578179,-2.361891,9.542182e+05,9.763811e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,4.091104,6.599758,1.040911e+06,1.065998e+06


In [10]:
serial_vs_serial_diffs = compare_results(serial_1, serial_2, "serial_1", "serial_2")

PASS: serial_1 and serial_2 match within atol=1e-06, rtol=1e-06.


## 4. Test B: Serial vs `loky`, `n_jobs=1`

This is the direct test of the suspicious case. `n_jobs=1` should match the serial loop. If it does not, then either imports/state/data differ between the main process and the worker process, or the function is not pure.

In [11]:
loky_1 = run_loky(CASES, n_jobs=1)
loky_1

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:  1.5min
[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:  2.9min
[Parallel(n_jobs=1)]: Done   3 tasks      | elapsed:  5.3min
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:  9.0min
[Parallel(n_jobs=1)]: Done   5 tasks      | elapsed: 12.8min
[Parallel(n_jobs=1)]: Done   6 tasks      | elapsed: 16.8min
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed: 17.6min
[Parallel(n_jobs=1)]: Done   8 tasks      | elapsed: 20.4min
[Parallel(n_jobs=1)]: Done   9 tasks      | elapsed: 22.2min
[Parallel(n_jobs=1)]: Done  10 tasks      | elapsed: 23.9min
[Parallel(n_jobs=1)]: Done  11 tasks      | elapsed: 26.3min
[Parallel(n_jobs=1)]: Done  11 out of  11 | elapsed: 26.3min finished
loky n_jobs=1 elapsed: 26.3 min


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469640,11.623609,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265471,7.624570,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,15.132485,17.690278,1.151325e+06,1.176903e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,16.317112,19.003864,1.163171e+06,1.190039e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-14.717779,-11.835155,8.528222e+05,8.816485e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-2.459850,0.296069,9.754015e+05,1.002961e+06
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-4.505327,-3.368760,9.549467e+05,9.663124e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-10.174324,-9.274388,8.982568e+05,9.072561e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-4.578179,-2.361891,9.542182e+05,9.763811e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,4.091104,6.599758,1.040911e+06,1.065998e+06


In [12]:
serial_vs_loky1_diffs = compare_results(serial_1, loky_1, "serial", "loky1")

PASS: serial and loky1 match within atol=1e-06, rtol=1e-06.


## 5. Test C: Serial vs Multi-worker `loky`

Only run this after Test A and Test B pass. If `n_jobs=1` passes but `n_jobs>1` fails, the issue is specifically related to concurrent execution, memory pressure, nested multiprocessing, or shared external resources.

In [13]:
# Adjust to your machine. Start with 2 before trying 4 or -1.
loky_2 = run_loky(CASES, n_jobs=2)
loky_2

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:  1.1min
[Parallel(n_jobs=2)]: Done   2 tasks      | elapsed:  1.5min
[Parallel(n_jobs=2)]: Done   3 tasks      | elapsed:  2.9min
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:  5.2min
[Parallel(n_jobs=2)]: Done   5 tasks      | elapsed:  7.0min
[Parallel(n_jobs=2)]: Done   6 tasks      | elapsed:  7.9min
[Parallel(n_jobs=2)]: Done   7 tasks      | elapsed:  9.5min
[Parallel(n_jobs=2)]: Done   8 tasks      | elapsed: 10.4min
[Parallel(n_jobs=2)]: Done   9 out of  11 | elapsed: 10.9min remaining:  2.4min
[Parallel(n_jobs=2)]: Done  11 out of  11 | elapsed: 14.0min finished
loky n_jobs=2 elapsed: 14.0 min


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469640,11.623609,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265503,7.624602,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,18.157186,20.901080,1.181572e+06,1.209011e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,19.047044,21.474025,1.190470e+06,1.214740e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-9.586433,-6.924404,9.041357e+05,9.307560e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,5.283632,8.043399,1.052836e+06,1.080434e+06
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-12.334374,-11.323423,8.766563e+05,8.867658e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-11.122029,-10.310639,8.887797e+05,8.968936e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-10.175979,-7.937062,8.982402e+05,9.206294e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,2.903817,5.476804,1.029038e+06,1.054768e+06


In [14]:
serial_vs_loky2_diffs = compare_results(serial_1, loky_2, "serial", "loky2")

FAIL: serial and loky2 differ.
DataFrame.iloc[:, 9] (column name="net_return_pct") are different

DataFrame.iloc[:, 9] (column name="net_return_pct") values are different (90.90909 %)
[index]: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[left]:  [9.469640048829863, 5.265471350674522, 15.132485282869723, 16.317111658729665, -14.717778614663391, -2.4598498778550693, -4.505326878323801, -10.17432420678871, -4.578178582125892, 4.09110396026664, -5.38587371152397]
[right]: [9.469640246022548, 5.265502725601423, 18.157186402001724, 19.04704400245678, -9.586432806954425, 5.283631672467504, -12.334373725215041, -11.122028779122406, -10.175979497925914, 2.9038173901477116, -6.842702050090432]
At positional index 1, first diff: 5.265471350674522 != 5.265502725601423


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,...,gross_return_pct_serial,gross_return_pct_loky2,gross_return_pct_diff,final_net_equity_serial,final_net_equity_loky2,final_net_equity_diff,final_gross_equity_serial,final_gross_equity_loky2,final_gross_equity_diff,_merge
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,...,11.623609,11.623609,6.793406e-08,1.094696e+06,1.094696e+06,0.001972,1.116236e+06,1.116236e+06,0.000679,both
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,...,7.624570,7.624602,3.170867e-05,1.052655e+06,1.052655e+06,0.313749,1.076246e+06,1.076246e+06,0.317087,both
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,...,17.690278,20.901080,3.210801e+00,1.151325e+06,1.181572e+06,30247.011191,1.176903e+06,1.209011e+06,32108.011714,both
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,...,19.003864,21.474025,2.470161e+00,1.163171e+06,1.190470e+06,27299.323437,1.190039e+06,1.214740e+06,24701.612187,both
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,...,-11.835155,-6.924404,4.910751e+00,8.528222e+05,9.041357e+05,51313.458077,8.816485e+05,9.307560e+05,49107.510889,both
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,...,0.296069,8.043399,7.747329e+00,9.754015e+05,1.052836e+06,77434.815503,1.002961e+06,1.080434e+06,77473.294281,both
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,...,-3.368760,-11.323423,-7.954663e+00,9.549467e+05,8.766563e+05,-78290.468469,9.663124e+05,8.867658e+05,-79546.632314,both
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,...,-9.274388,-10.310639,-1.036251e+00,8.982568e+05,8.887797e+05,-9477.045723,9.072561e+05,8.968936e+05,-10362.510961,both
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,...,-2.361891,-7.937062,-5.575171e+00,9.542182e+05,8.982402e+05,-55978.009158,9.763811e+05,9.206294e+05,-55751.708959,both
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,...,6.599758,5.476804,-1.122954e+00,1.040911e+06,1.029038e+06,-11872.865701,1.065998e+06,1.054768e+06,-11229.535882,both


## 6. Strict Frozen-input Test

The previous tests use the current `run_grid_case`, which downloads data inside each case. This section freezes the raw Yahoo OHLCV data once, then gives every case the same raw input.

This is the cleaner design for parallel research: workers should not independently download external data.

In [15]:
def download_frozen_raw(fixed=FIXED, cases=CASES):
    max_training_years = max(case["training_years"] for case in cases)
    params = {
        "symbol": fixed["symbol"],
        "start_date": fixed["start_date"],
        "end_date": fixed["end_date"],
        "training_years": max_training_years,
        "signal_threshold": fixed["signal_threshold"],
        "n_features": fixed.get("n_features", 6),
        "max_iter": fixed["max_iter"],
        "verbose": fixed["verbose"],
    }
    strategy = MLPredictorStrategy(params=params)
    raw = strategy.download_dataset(training_years=max_training_years)
    return raw.copy(deep=True)


def prepare_dataset_from_raw(raw_ohlcv, params):
    strategy = MLPredictorStrategy(params=params)
    df = raw_ohlcv.copy(deep=True)
    df = strategy.assign_signal(df)
    df = strategy.compute_technical_indicators(df).copy()
    df = strategy.clean_dataset(df)
    return strategy, df


def evaluate_case_frozen(case_id, case, raw_ohlcv, fixed=FIXED):
    params = {**fixed, **case}
    strategy, dataset = prepare_dataset_from_raw(raw_ohlcv, params)
    weights = strategy.generate_signals(dataset, show_progress=params["show_progress"])
    if weights.empty:
        raise RuntimeError(f"No weights produced for case_id={case_id}, params={params}")

    prices = dataset[["Open"]].rename(columns={"Open": params["symbol"]}).loc[weights.index]
    engine = PortfolioVectorEngine(
        prices=prices,
        signals=weights,
        asset_class=params["asset_class"],
        initial_capital=params["initial_capital"],
        execution_delay=params["execution_delay"],
    )
    stats = engine.run()

    return {
        "case_id": case_id,
        "symbol": params["symbol"],
        "start_date": params["start_date"],
        "end_date": params["end_date"],
        "signal_threshold": params["signal_threshold"],
        "n_features": params["n_features"],
        "training_years": params["training_years"],
        "position_size": params["position_size"],
        "max_iter": params["max_iter"],
        "execution_delay": params["execution_delay"],
        "net_return_pct": (stats["net_equity"].iloc[-1] / params["initial_capital"] - 1) * 100,
        "gross_return_pct": (stats["gross_equity"].iloc[-1] / params["initial_capital"] - 1) * 100,
        "final_net_equity": stats["net_equity"].iloc[-1],
        "final_gross_equity": stats["gross_equity"].iloc[-1],
    }


def run_serial_frozen(raw_ohlcv, cases=CASES):
    records = []
    start = time.time()
    for case_id, case in enumerate(cases):
        print(f"serial frozen case {case_id}: {case}")
        records.append(evaluate_case_frozen(case_id, case, raw_ohlcv))
    print(f"serial frozen elapsed: {(time.time() - start) / 60:.1f} min")
    return normalize_results(records)


def run_loky_frozen(raw_ohlcv, cases=CASES, n_jobs=1):
    get_reusable_executor().shutdown(wait=True)
    start = time.time()
    records = Parallel(n_jobs=n_jobs, backend="loky", verbose=50)(
        delayed(evaluate_case_frozen)(case_id, case, raw_ohlcv)
        for case_id, case in enumerate(cases)
    )
    print(f"loky frozen n_jobs={n_jobs} elapsed: {(time.time() - start) / 60:.1f} min")
    return normalize_results(records)

In [16]:
raw_ohlcv = download_frozen_raw(FIXED, CASES)
raw_ohlcv.head(), raw_ohlcv.tail(), raw_ohlcv.shape

(ECH              Open       High        Low      Close  Volume
 Date                                                          
 2010-01-04  36.720272  37.202651  36.356837  37.070492   91800
 2010-01-05  37.242304  37.367854  37.070496  37.222481  122300
 2010-01-06  37.513229  37.883274  37.427326  37.823803  118500
 2010-01-07  37.810599  38.709278  37.764343  38.709278  653300
 2010-01-08  38.788556  39.284151  38.610142  39.237896  336000,
 ECH              Open       High        Low      Close  Volume
 Date                                                          
 2025-06-24  29.752461  30.468561  29.752461  30.370464  416400
 2025-06-25  30.419513  30.497989  30.252750  30.272369   83300
 2025-06-26  30.635322  30.880561  30.596084  30.664751  264200
 2025-06-27  30.576464  30.743227  30.262557  30.458748  207300
 2025-06-30  30.390084  30.939422  30.390084  30.851135  234000,
 (3896, 5))

In [17]:
frozen_serial = run_serial_frozen(raw_ohlcv, CASES)
frozen_serial

serial frozen case 0: {'n_features': 6, 'training_years': 10}
serial frozen case 1: {'n_features': 8, 'training_years': 10}
serial frozen case 2: {'n_features': 10, 'training_years': 10}
serial frozen case 3: {'n_features': 12, 'training_years': 10}
serial frozen case 4: {'n_features': 14, 'training_years': 10}
serial frozen case 5: {'n_features': 16, 'training_years': 10}
serial frozen case 6: {'n_features': 12, 'training_years': 4}
serial frozen case 7: {'n_features': 12, 'training_years': 6}
serial frozen case 8: {'n_features': 12, 'training_years': 8}
serial frozen case 9: {'n_features': 12, 'training_years': 12}
serial frozen case 10: {'n_features': 12, 'training_years': 14}
serial frozen elapsed: 24.2 min


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469624,11.623593,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265458,7.624557,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,17.604645,20.076748,1.176046e+06,1.200767e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,15.531589,18.106538,1.155316e+06,1.181065e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-4.270888,-1.490015,9.572911e+05,9.850999e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-10.515531,-7.693842,8.948447e+05,9.230616e+05
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-9.882834,-8.826552,9.011717e+05,9.117345e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-8.726844,-7.808892,9.127316e+05,9.219111e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-5.839262,-3.561007,9.416074e+05,9.643899e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,2.055862,4.545632,1.020559e+06,1.045456e+06


In [18]:
frozen_loky1 = run_loky_frozen(raw_ohlcv, CASES, n_jobs=1)
frozen_loky1

[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:  1.3min
[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:  2.4min
[Parallel(n_jobs=1)]: Done   3 tasks      | elapsed:  4.5min
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:  8.7min
[Parallel(n_jobs=1)]: Done   5 tasks      | elapsed: 12.7min
[Parallel(n_jobs=1)]: Done   6 tasks      | elapsed: 17.6min
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed: 18.6min
[Parallel(n_jobs=1)]: Done   8 tasks      | elapsed: 21.1min
[Parallel(n_jobs=1)]: Done   9 tasks      | elapsed: 22.4min
[Parallel(n_jobs=1)]: Done  10 tasks      | elapsed: 24.3min
[Parallel(n_jobs=1)]: Done  11 tasks      | elapsed: 27.1min
[Parallel(n_jobs=1)]: Done  11 out of  11 | elapsed: 27.1min finished
loky frozen n_jobs=1 elapsed: 27.1 min


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469624,11.623593,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265458,7.624557,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,17.604645,20.076748,1.176046e+06,1.200767e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,15.531589,18.106538,1.155316e+06,1.181065e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-4.270888,-1.490015,9.572911e+05,9.850999e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-10.515531,-7.693842,8.948447e+05,9.230616e+05
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-9.882834,-8.826552,9.011717e+05,9.117345e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-8.726844,-7.808892,9.127316e+05,9.219111e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-5.839262,-3.561007,9.416074e+05,9.643899e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,2.055862,4.545632,1.020559e+06,1.045456e+06


In [19]:
frozen_serial_vs_loky1_diffs = compare_results(frozen_serial, frozen_loky1, "frozen_serial", "frozen_loky1")

PASS: frozen_serial and frozen_loky1 match within atol=1e-06, rtol=1e-06.


In [20]:
# Optional after the previous frozen test passes.
frozen_loky2 = run_loky_frozen(raw_ohlcv, CASES, n_jobs=2)
frozen_serial_vs_loky2_diffs = compare_results(frozen_serial, frozen_loky2, "frozen_serial", "frozen_loky2")

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:  1.5min
[Parallel(n_jobs=2)]: Done   2 tasks      | elapsed:  1.8min
[Parallel(n_jobs=2)]: Done   3 tasks      | elapsed:  3.7min
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:  5.8min
[Parallel(n_jobs=2)]: Done   5 tasks      | elapsed:  8.0min
[Parallel(n_jobs=2)]: Done   6 tasks      | elapsed:  9.0min
[Parallel(n_jobs=2)]: Done   7 tasks      | elapsed: 10.5min
[Parallel(n_jobs=2)]: Done   8 tasks      | elapsed: 12.0min
[Parallel(n_jobs=2)]: Done   9 out of  11 | elapsed: 12.2min remaining:  2.7min
[Parallel(n_jobs=2)]: Done  11 out of  11 | elapsed: 14.6min finished
loky frozen n_jobs=2 elapsed: 14.6 min
PASS: frozen_serial and frozen_loky2 match within atol=1e-06, rtol=1e-06.


In [21]:
frozen_loky2

,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469624,11.623593,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265458,7.624557,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,17.604645,20.076748,1.176046e+06,1.200767e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,15.531589,18.106538,1.155316e+06,1.181065e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-4.270888,-1.490015,9.572911e+05,9.850999e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-10.515531,-7.693842,8.948447e+05,9.230616e+05
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-9.882834,-8.826552,9.011717e+05,9.117345e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-8.726844,-7.808892,9.127316e+05,9.219111e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-5.839262,-3.561007,9.416074e+05,9.643899e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,2.055862,4.545632,1.020559e+06,1.045456e+06


In [22]:
# Optional after the previous frozen test passes.
frozen_loky4 = run_loky_frozen(raw_ohlcv, CASES, n_jobs=4)
frozen_serial_vs_loky4_diffs = compare_results(frozen_serial, frozen_loky4, "frozen_serial", "frozen_loky4")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:  2.1min
[Parallel(n_jobs=4)]: Done   2 tasks      | elapsed:  2.5min
[Parallel(n_jobs=4)]: Done   3 tasks      | elapsed:  3.0min
[Parallel(n_jobs=4)]: Done   4 tasks      | elapsed:  5.0min
[Parallel(n_jobs=4)]: Done   5 out of  11 | elapsed:  6.9min remaining:  8.3min
[Parallel(n_jobs=4)]: Done   6 out of  11 | elapsed:  9.1min remaining:  7.6min
[Parallel(n_jobs=4)]: Done   7 out of  11 | elapsed:  9.3min remaining:  5.3min
[Parallel(n_jobs=4)]: Done   8 out of  11 | elapsed:  9.3min remaining:  3.5min
[Parallel(n_jobs=4)]: Done   9 out of  11 | elapsed:  9.9min remaining:  2.2min
[Parallel(n_jobs=4)]: Done  11 out of  11 | elapsed: 11.7min finished
loky frozen n_jobs=4 elapsed: 11.7 min
PASS: frozen_serial and frozen_loky4 match within atol=1e-06, rtol=1e-06.


In [23]:
frozen_loky4

,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469624,11.623593,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265458,7.624557,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,17.604645,20.076748,1.176046e+06,1.200767e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,15.531589,18.106538,1.155316e+06,1.181065e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-4.270888,-1.490015,9.572911e+05,9.850999e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-10.515531,-7.693842,8.948447e+05,9.230616e+05
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-9.882834,-8.826552,9.011717e+05,9.117345e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-8.726844,-7.808892,9.127316e+05,9.219111e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-5.839262,-3.561007,9.416074e+05,9.643899e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,2.055862,4.545632,1.020559e+06,1.045456e+06


In [24]:
# Optional after the previous frozen test passes.
frozen_lokyN1 = run_loky_frozen(raw_ohlcv, CASES, n_jobs=-1)
frozen_serial_vs_lokyN1_diffs = compare_results(frozen_serial, frozen_lokyN1, "frozen_serial", "frozen_lokyN1")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done   2 out of  11 | elapsed:  3.9min remaining: 17.7min
[Parallel(n_jobs=-1)]: Done   3 out of  11 | elapsed:  4.8min remaining: 12.8min
[Parallel(n_jobs=-1)]: Done   4 out of  11 | elapsed:  6.6min remaining: 11.6min
[Parallel(n_jobs=-1)]: Done   5 out of  11 | elapsed:  7.6min remaining:  9.1min
[Parallel(n_jobs=-1)]: Done   6 out of  11 | elapsed:  8.1min remaining:  6.8min
[Parallel(n_jobs=-1)]: Done   7 out of  11 | elapsed:  8.8min remaining:  5.1min
[Parallel(n_jobs=-1)]: Done   8 out of  11 | elapsed: 10.1min remaining:  3.8min
[Parallel(n_jobs=-1)]: Done   9 out of  11 | elapsed: 10.2min remaining:  2.3min
[Parallel(n_jobs=-1)]: Done  11 out of  11 | elapsed: 11.1min finished
loky frozen n_jobs=-1 elapsed: 11.1 min
PASS: frozen_serial and frozen_lokyN1 match within atol=1e-06, rtol=1e-06.


In [25]:
frozen_lokyN1

,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,net_return_pct,gross_return_pct,final_net_equity,final_gross_equity
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,9.469624,11.623593,1.094696e+06,1.116236e+06
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,5.265458,7.624557,1.052655e+06,1.076246e+06
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,17.604645,20.076748,1.176046e+06,1.200767e+06
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,15.531589,18.106538,1.155316e+06,1.181065e+06
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,-4.270888,-1.490015,9.572911e+05,9.850999e+05
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,-10.515531,-7.693842,8.948447e+05,9.230616e+05
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,-9.882834,-8.826552,9.011717e+05,9.117345e+05
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,-8.726844,-7.808892,9.127316e+05,9.219111e+05
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,-5.839262,-3.561007,9.416074e+05,9.643899e+05
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,2.055862,4.545632,1.020559e+06,1.045456e+06


#### All the runs with frozen-input data (frozen_serial, frozen_loky1, ...) give IDENTICAL results

> \>  Need to use **saved data**

## 7. Test D: Fresh-input vs Frozen-input



In [26]:
serial_vs_frozen_serial_diffs = compare_results(serial_1, frozen_serial, "serial", "frozen_serial") 

FAIL: serial and frozen_serial differ.
DataFrame.iloc[:, 9] (column name="net_return_pct") are different

DataFrame.iloc[:, 9] (column name="net_return_pct") values are different (90.90909 %)
[index]: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[left]:  [9.469640048829863, 5.265471350674522, 15.132485282869723, 16.317111658729665, -14.717778614663391, -2.4598498778550693, -4.505326878323801, -10.17432420678871, -4.578178582125892, 4.09110396026664, -5.38587371152397]
[right]: [9.46962402749274, 5.26545815206787, 17.60464545093614, 15.531589302925486, -4.270887808802426, -10.515531028123249, -9.882834437452226, -8.726844488057207, -5.8392615506813295, 2.055861536876824, -5.38587371152397]
At positional index 0, first diff: 9.469640048829863 != 9.46962402749274


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,...,gross_return_pct_serial,gross_return_pct_frozen_serial,gross_return_pct_diff,final_net_equity_serial,final_net_equity_frozen_serial,final_net_equity_diff,final_gross_equity_serial,final_gross_equity_frozen_serial,final_gross_equity_diff,_merge
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,...,11.623609,11.623593,-0.000016,1.094696e+06,1.094696e+06,-0.160213,1.116236e+06,1.116236e+06,-0.162820,both
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,...,7.624570,7.624557,-0.000013,1.052655e+06,1.052655e+06,-0.131986,1.076246e+06,1.076246e+06,-0.133897,both
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,...,17.690278,20.076748,2.386470,1.151325e+06,1.176046e+06,24721.601681,1.176903e+06,1.200767e+06,23864.696565,both
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,...,19.003864,18.106538,-0.897326,1.163171e+06,1.155316e+06,-7855.223558,1.190039e+06,1.181065e+06,-8973.264806,both
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,...,-11.835155,-1.490015,10.345140,8.528222e+05,9.572911e+05,104468.908059,8.816485e+05,9.850999e+05,103451.399130,both
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,...,0.296069,-7.693842,-7.989912,9.754015e+05,8.948447e+05,-80556.811503,1.002961e+06,9.230616e+05,-79899.115320,both
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,...,-3.368760,-8.826552,-5.457792,9.549467e+05,9.011717e+05,-53775.075591,9.663124e+05,9.117345e+05,-54577.917773,both
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,...,-9.274388,-7.808892,1.465496,8.982568e+05,9.127316e+05,14474.797187,9.072561e+05,9.219111e+05,14654.959588,both
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,...,-2.361891,-3.561007,-1.199116,9.542182e+05,9.416074e+05,-12610.829686,9.763811e+05,9.643899e+05,-11991.161605,both
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,...,6.599758,4.545632,-2.054125,1.040911e+06,1.020559e+06,-20352.424234,1.065998e+06,1.045456e+06,-20541.251501,both


In [35]:
loky2_vs_frozen_serial_diffs = compare_results(loky_2, frozen_serial, "loky_2", "frozen_serial")

FAIL: loky_2 and frozen_serial differ.
DataFrame.iloc[:, 9] (column name="net_return_pct") are different

DataFrame.iloc[:, 9] (column name="net_return_pct") values are different (100.0 %)
[index]: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[left]:  [9.469640246022548, 5.265502725601423, 18.157186402001724, 19.04704400245678, -9.586432806954425, 5.283631672467504, -12.334373725215041, -11.122028779122406, -10.175979497925914, 2.9038173901477116, -6.842702050090432]
[right]: [9.46962402749274, 5.26545815206787, 17.60464545093614, 15.531589302925486, -4.270887808802426, -10.515531028123249, -9.882834437452226, -8.726844488057207, -5.8392615506813295, 2.055861536876824, -5.38587371152397]
At positional index 0, first diff: 9.469640246022548 != 9.46962402749274


,case_id,symbol,start_date,end_date,signal_threshold,n_features,training_years,position_size,max_iter,execution_delay,...,gross_return_pct_loky_2,gross_return_pct_frozen_serial,gross_return_pct_diff,final_net_equity_loky_2,final_net_equity_frozen_serial,final_net_equity_diff,final_gross_equity_loky_2,final_gross_equity_frozen_serial,final_gross_equity_diff,_merge
0,0,ECH,2025-01-01,2025-07-01,0.002,6,10,1.0,5000,1,...,11.623609,11.623593,-0.000016,1.094696e+06,1.094696e+06,-0.162185,1.116236e+06,1.116236e+06,-0.163499,both
1,1,ECH,2025-01-01,2025-07-01,0.002,8,10,1.0,5000,1,...,7.624602,7.624557,-0.000045,1.052655e+06,1.052655e+06,-0.445735,1.076246e+06,1.076246e+06,-0.450984,both
2,2,ECH,2025-01-01,2025-07-01,0.002,10,10,1.0,5000,1,...,20.901080,20.076748,-0.824332,1.181572e+06,1.176046e+06,-5525.409511,1.209011e+06,1.200767e+06,-8243.315149,both
3,3,ECH,2025-01-01,2025-07-01,0.002,12,10,1.0,5000,1,...,21.474025,18.106538,-3.367488,1.190470e+06,1.155316e+06,-35154.546995,1.214740e+06,1.181065e+06,-33674.876992,both
4,4,ECH,2025-01-01,2025-07-01,0.002,14,10,1.0,5000,1,...,-6.924404,-1.490015,5.434389,9.041357e+05,9.572911e+05,53155.449982,9.307560e+05,9.850999e+05,54343.888241,both
5,5,ECH,2025-01-01,2025-07-01,0.002,16,10,1.0,5000,1,...,8.043399,-7.693842,-15.737241,1.052836e+06,8.948447e+05,-157991.627006,1.080434e+06,9.230616e+05,-157372.409601,both
6,6,ECH,2025-01-01,2025-07-01,0.002,12,4,1.0,5000,1,...,-11.323423,-8.826552,2.496871,8.766563e+05,9.011717e+05,24515.392878,8.867658e+05,9.117345e+05,24968.714541,both
7,7,ECH,2025-01-01,2025-07-01,0.002,12,6,1.0,5000,1,...,-10.310639,-7.808892,2.501747,8.887797e+05,9.127316e+05,23951.842911,8.968936e+05,9.219111e+05,25017.470550,both
8,8,ECH,2025-01-01,2025-07-01,0.002,12,8,1.0,5000,1,...,-7.937062,-3.561007,4.376055,8.982402e+05,9.416074e+05,43367.179472,9.206294e+05,9.643899e+05,43760.547354,both
9,9,ECH,2025-01-01,2025-07-01,0.002,12,12,1.0,5000,1,...,5.476804,4.545632,-0.931172,1.029038e+06,1.020559e+06,-8479.558533,1.054768e+06,1.045456e+06,-9311.715618,both


All the other comparisons can be deduced from the obtained comparisons.

## How To Interpret Results

- If `serial_1` differs from `serial_2`, the backtest is nondeterministic even without parallelism.
- If serial matches serial but differs from `loky_1`, the issue is process/import/state isolation. Restarting the kernel and shutting down reusable workers should be mandatory before research runs.
- If `loky_1` matches but `loky_2` differs, the issue is concurrent execution, external data downloads, memory pressure, or nested multiprocessing.
- If the frozen-input tests pass, then the safest research design is to freeze inputs first and parallelize only pure case evaluation.